In [1]:
# !pip install -U ollama

In [2]:
queries = [
    "show me citations for the paper \"Attention Is All You Need\" by Vaswani et al 2017",
    "list all papers by Yoshua Bengio presented at neurips between 2015 and 2020",
    'who are the authors of the iclr 2021 paper "self-supervised learning is all you need"?',
    "give me the affiliation of the author named Raquel Urtasun in icml publications",
    "list all papers by Michael Jordan at nips between 1995 and 2005",
    'who are the authors of "DBLPLink 2.0 - An Entity Linker for the DBLP Scholarly Knowledge Graph"'
]

data = { q: {} for q in queries }

In [3]:
import re
from SPARQLWrapper import SPARQLWrapper, JSON

def sanitize_query(q: str) -> str:
    # remove ```sparql ... ``` or ``` ... ```
    q = re.sub(r"^```(?:sparql)?\s*", "", q.strip(), flags=re.IGNORECASE)
    q = re.sub(r"\s*```$", "", q, flags=re.MULTILINE)
    return q.strip()

def run(endpoint: str, query: str):
    query = sanitize_query(query)
    s = SPARQLWrapper(endpoint)
    s.setMethod("POST")
    s.setTimeout(60)
    s.setReturnFormat(JSON)
    s.addParameter("format", "json")
    s.addCustomHttpHeader("User-Agent", "NL2SPARQL/0.1 (jeffry.cacho@rwth-aachen.de)")
    s.setQuery(query)
    return s.query().convert()["results"]["bindings"]


In [4]:
import os
import json
from ollama import AsyncClient
from dotenv import load_dotenv
load_dotenv()

client = AsyncClient(
  host='http://ollama.warhol.informatik.rwth-aachen.de',
  headers={'x-api-key': os.getenv('OLLAMA_API_KEY')}
)
res = await client.list()

for model in res.models:
    print(model)

model='qwen3-embedding:8b' modified_at=datetime.datetime(2026, 2, 24, 7, 20, 18, 390960, tzinfo=TzInfo(0)) digest='64b933495768fbd3b87c20583d379728a07471e0c66733a9df87cd1901b3c44b' size=4676805193 details=ModelDetails(parent_model='', format='gguf', family='qwen3', families=['qwen3'], parameter_size='7.6B', quantization_level='Q4_K_M')
model='qwen3-coder:30b' modified_at=datetime.datetime(2026, 2, 10, 14, 6, 46, 436637, tzinfo=TzInfo(0)) digest='06c1097efce0431c2045fe7b2e5108366e43bee1b4603a7aded8f21689e90bca' size=18556700761 details=ModelDetails(parent_model='', format='gguf', family='qwen3moe', families=['qwen3moe'], parameter_size='30.5B', quantization_level='Q4_K_M')
model='llama3.3:70b' modified_at=datetime.datetime(2026, 2, 3, 9, 24, 45, 32404, tzinfo=TzInfo(0)) digest='a6eb4748fd2990ad2952b2335a95a7f952d1a06119a0aa6a2df6cd052a93a3fa' size=42520413916 details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='70.6B', quantization_lev

# Load context Pack

In [6]:
ENDPOINT = "https://sparql.dblp.org/sparql"

In [ ]:
from urllib.parse import unquote
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Set, Tuple

from rdflib import Graph, Literal, URIRef
from rdflib.namespace import OWL, RDF, RDFS, XSD


# ----------------------------
# Data model
# ----------------------------

@dataclass
class ClassDef:
    iri: str
    labels: List[str] = field(default_factory=list)
    comments: List[str] = field(default_factory=list)

    super_classes: Set[str] = field(default_factory=set)        # rdfs:subClassOf
    equivalent_classes: Set[str] = field(default_factory=set)   # owl:equivalentClass
    disjoint_with: Set[str] = field(default_factory=set)        # owl:disjointWith


@dataclass
class PropDef:
    iri: str
    labels: List[str] = field(default_factory=list)
    comments: List[str] = field(default_factory=list)

    domains: Set[str] = field(default_factory=set)              # rdfs:domain
    literal_datatypes: Set[str] = field(default_factory=set)    # rdfs:range in XSD namespace
    range_classes: Set[str] = field(default_factory=set)        # rdfs:range for non XSD IRIs

    super_properties: Set[str] = field(default_factory=set)     # rdfs:subPropertyOf
    equivalent_properties: Set[str] = field(default_factory=set)# owl:equivalentProperty
    inverses: Set[str] = field(default_factory=set)             # owl:inverseOf
    property_types: Set[str] = field(default_factory=set)       # rdf:type values like owl:ObjectProperty etc


@dataclass
class SchemaIndex:
    namespaces: Dict[str, str] = field(default_factory=dict)    # prefix -> namespace
    classes: Dict[str, ClassDef] = field(default_factory=dict)  # iri -> def
    props: Dict[str, PropDef] = field(default_factory=dict)     # iri -> def


# ----------------------------
# CURIE helpers
# ----------------------------

def _ensure_default_prefixes(namespaces: Dict[str, str]) -> Dict[str, str]:
    defaults = {"rdf": str(RDF), "rdfs": str(RDFS), "owl": str(OWL), "xsd": str(XSD)}
    out = dict(namespaces)
    for k, v in defaults.items():
        out.setdefault(k, v)
    return out


def make_curie(iri: str, namespaces: Dict[str, str]) -> str:
    iri_u = unquote(iri)
    best: Optional[Tuple[str, str]] = None
    best_len = -1
    for pfx, ns in namespaces.items():
        ns_u = unquote(ns)
        if iri_u.startswith(ns_u) and len(ns_u) > best_len:
            best = (pfx, ns_u)
            best_len = len(ns_u)
    if best is None:
        return iri
    pfx, ns_u = best
    return f"{pfx}:{iri_u[best_len:]}"


# ----------------------------
# Load schema
# ----------------------------

def load_schema(path: str, base_iri: Optional[str] = None) -> SchemaIndex:
    g = Graph()
    if base_iri is None:
        g.parse(path)
    else:
        g.parse(path, publicID=base_iri)

    ns: Dict[str, str] = {pfx: str(uri) for pfx, uri in g.namespaces()}
    ns = _ensure_default_prefixes(ns)
    idx = SchemaIndex(namespaces=ns)

    # -------- classes --------
    class_nodes: Set[URIRef] = set()
    for s in g.subjects(RDF.type, OWL.Class):
        if isinstance(s, URIRef):
            class_nodes.add(s)
    for s in g.subjects(RDF.type, RDFS.Class):
        if isinstance(s, URIRef):
            class_nodes.add(s)

    for c in class_nodes:
        iri = str(c)
        cd = idx.classes.get(iri) or ClassDef(iri=iri)

        cd.labels = _collect_literals(g, c, RDFS.label)
        cd.comments = _collect_literals(g, c, RDFS.comment)

        cd.super_classes |= {str(o) for o in g.objects(c, RDFS.subClassOf) if isinstance(o, URIRef)}
        cd.equivalent_classes |= {str(o) for o in g.objects(c, OWL.equivalentClass) if isinstance(o, URIRef)}
        cd.disjoint_with |= {str(o) for o in g.objects(c, OWL.disjointWith) if isinstance(o, URIRef)}

        idx.classes[iri] = cd

    # -------- properties --------
    prop_nodes: Set[URIRef] = set()

    # collect any node that is declared a property via type
    property_type_iris = {
        RDF.Property,
        OWL.ObjectProperty,
        OWL.DatatypeProperty,
        OWL.AnnotationProperty,
        OWL.OntologyProperty,
        OWL.FunctionalProperty,
        OWL.InverseFunctionalProperty,
        OWL.SymmetricProperty,
        OWL.TransitiveProperty,
    }
    for t in property_type_iris:
        for s in g.subjects(RDF.type, t):
            if isinstance(s, URIRef):
                prop_nodes.add(s)

    # also include anything used as a predicate in the graph (schema files often omit explicit typing)
    for p in set(g.predicates()):
        if isinstance(p, URIRef):
            prop_nodes.add(p)

    for p in prop_nodes:
        iri = str(p)
        pd = idx.props.get(iri) or PropDef(iri=iri)

        pd.labels = _collect_literals(g, p, RDFS.label)
        pd.comments = _collect_literals(g, p, RDFS.comment)

        # all rdf:type signals (not just object/datatype)
        for t in g.objects(p, RDF.type):
            if isinstance(t, URIRef):
                pd.property_types.add(str(t))

        pd.domains |= {str(o) for o in g.objects(p, RDFS.domain) if isinstance(o, URIRef)}

        for o in g.objects(p, RDFS.range):
            if not isinstance(o, URIRef):
                continue
            o_str = str(o)
            if o_str.startswith(str(XSD)):
                pd.literal_datatypes.add(o_str)
            else:
                pd.range_classes.add(o_str)

        pd.super_properties |= {str(o) for o in g.objects(p, RDFS.subPropertyOf) if isinstance(o, URIRef)}
        pd.equivalent_properties |= {str(o) for o in g.objects(p, OWL.equivalentProperty) if isinstance(o, URIRef)}

        # owl:inverseOf is symmetric in practice, store both directions
        for o in g.objects(p, OWL.inverseOf):
            if isinstance(o, URIRef):
                pd.inverses.add(str(o))

        idx.props[iri] = pd

    # second pass: make inverseOf explicitly symmetric in the index
    for piri, pd in list(idx.props.items()):
        for inv in list(pd.inverses):
            inv_pd = idx.props.get(inv)
            if inv_pd is None:
                inv_pd = PropDef(iri=inv)
                idx.props[inv] = inv_pd
            inv_pd.inverses.add(piri)

    return idx


def _collect_literals(g: Graph, s: URIRef, p: URIRef) -> List[str]:
    out: List[str] = []
    for o in g.objects(s, p):
        if isinstance(o, Literal):
            val = str(o).strip()
            if val and val not in out:
                out.append(val)
    return out

In [ ]:
from typing import List, Dict, Optional, Iterable

# ----------------------------
# Compact context renderer
# ----------------------------

def _range_compact(p: PropDef, namespaces: Dict[str, str]) -> str:
    if p.literal_datatypes:
        dts = sorted(make_curie(dt, namespaces) for dt in p.literal_datatypes)
        return f"lit({','.join(dts)})"
    if p.range_classes:
        cs = sorted(make_curie(c, namespaces) for c in p.range_classes)
        return f"iri({'|'.join(cs)})"
    return "unk"


def _fmt_curie_list(iris: Iterable[str], namespaces: Dict[str, str], *, max_items: Optional[int] = None) -> str:
    items = [make_curie(i, namespaces) for i in iris]
    items = sorted(dict.fromkeys(items))  # stable dedupe
    if max_items is not None:
        items = items[:max_items]
    return ", ".join(items)


def schema_to_context_string(
    idx: SchemaIndex,
    *,
    include_prefixes: bool = True,
    max_types: Optional[int] = None,
    max_preds_per_type: Optional[int] = None,
    include_notes: bool = True,
    include_predicates: bool = True,
    include_type_header: bool = True,
    include_pred_header: bool = True,
    include_class_relations: bool = True,
    include_superclasses: bool = True,
    include_equivalents: bool = True,
    max_rel_items: Optional[int] = None,
) -> str:
    lines: List[str] = []

    if include_prefixes:
        lines.append("PREFIXES")
        for pfx, ns in sorted(idx.namespaces.items()):
            lines.append(f"{pfx}: <{ns}>")
        lines.append("")

    lines.append("SCHEMA")
    lines.append("Legend: type | label | opt(note) ; pred | label | rng: lit(xsd:*) or iri(Type)")
    if include_class_relations:
        lines.append("Legend2: subClassOf: ..., equiv: ... (direct links only)")
    lines.append("")

    # domain -> predicates
    domain_to_preds: Dict[str, List[PropDef]] = {}
    if include_predicates:
        for p in idx.props.values():
            for d in p.domains:
                domain_to_preds.setdefault(d, []).append(p)

    # sorted types
    types = sorted(idx.classes.values(), key=lambda c: make_curie(c.iri, idx.namespaces))
    if max_types is not None:
        types = types[:max_types]

    if include_type_header:
        lines.append("Types:")

    for c in types:
        c_id = make_curie(c.iri, idx.namespaces)
        c_label = c.labels[0] if c.labels else c_id.split(":")[-1]
        line = f"{c_id} | {c_label} | "
        if include_notes and c.comments:
            line += f"{';'.join(c.comments)}"
        lines.append(line)

        if include_class_relations:
            if include_superclasses:
                sups = getattr(c, "super_classes", set()) or set()
                if sups:
                    lines.append(f"  subClassOf: {_fmt_curie_list(sups, idx.namespaces, max_items=max_rel_items)}")

            if include_equivalents:
                eqs = getattr(c, "equivalent_classes", set()) or set()
                if eqs:
                    lines.append(f"  equiv: {_fmt_curie_list(eqs, idx.namespaces, max_items=max_rel_items)}")

        preds: List[PropDef] = []
        if include_predicates:
            preds = domain_to_preds.get(c.iri, [])
            preds.sort(key=lambda p: make_curie(p.iri, idx.namespaces))
            if max_preds_per_type is not None:
                preds = preds[:max_preds_per_type]

        if preds:
            if include_pred_header:
                lines.append("  Predicates:")
            for p in preds:
                p_id = make_curie(p.iri, idx.namespaces)
                p_label = p.labels[0] if p.labels else p_id.split(":")[-1]
                rng = _range_compact(p, idx.namespaces)
                lines.append(f"    {p_id} | {p_label} | rng:{rng}")

        lines.append("")

    return "\n".join(lines).rstrip()

In [9]:
from typing import Dict

def _is_xsd(dt_iri: str) -> bool:
    return dt_iri.startswith("http://www.w3.org/2001/XMLSchema#")


def _xsd_local(dt_iri: str) -> str:
    # http://www.w3.org/2001/XMLSchema#string -> string
    if "#" in dt_iri:
        return dt_iri.rsplit("#", 1)[-1]
    return dt_iri.rsplit("/", 1)[-1]


def filter_schema(
    idx: SchemaIndex,
    *,
    keep_only_literal_or_labelable_object: bool = False,
    include_predicates: bool = True,
    keep_only_classes_with_predicates: bool = False
) -> SchemaIndex:

    if not include_predicates:
        return SchemaIndex(
            namespaces=dict(idx.namespaces),
            classes=dict(idx.classes),
            props={},
        )

    # domain(class IRI) -> list[PropDef] (from original idx)
    domain_to_props: Dict[str, list] = {}
    for p in idx.props.values():
        for d in getattr(p, "domains", set()) or set():
            domain_to_props.setdefault(d, []).append(p)

    def class_is_labelable(class_iri: str) -> bool:
        for p2 in domain_to_props.get(class_iri, []):
            for dt in getattr(p2, "literal_datatypes", set()) or set():
                if _is_xsd(dt) and _xsd_local(dt).lower() == "string":
                    return True
        return False

    new_props: Dict[str, PropDef] = {}

    for p in idx.props.values():
        domains = getattr(p, "domains", set()) or set()
        if not domains:
            continue

        lit_dts = getattr(p, "literal_datatypes", set()) or set()
        rng_cls = getattr(p, "range_classes", set()) or set()

        has_lit = bool(lit_dts)
        has_obj = bool(rng_cls)

        if not has_lit and not has_obj:
            continue

        if keep_only_literal_or_labelable_object:
            if has_lit:
                keep = True
            else:
                keep = any(class_is_labelable(t) for t in rng_cls)
            if not keep:
                continue

        new_props[p.iri] = PropDef(
            iri=p.iri,
            labels=list(getattr(p, "labels", []) or []),
            comments=list(getattr(p, "comments", []) or []),
            domains=set(domains),
            literal_datatypes=set(lit_dts),
            range_classes=set(rng_cls),
        )

    # Filter classes if requested
    new_classes = dict(idx.classes)
    if keep_only_classes_with_predicates:
        # Collect all classes that appear as domains in the filtered properties
        classes_with_preds = set()
        for p in new_props.values():
            classes_with_preds.update(getattr(p, "domains", set()) or set())
        
        # Keep only those classes
        new_classes = {
            iri: cls_def 
            for iri, cls_def in idx.classes.items() 
            if iri in classes_with_preds
        }

    return SchemaIndex(
        namespaces=dict(idx.namespaces),
        classes=new_classes,
        props=new_props,
    )

In [10]:
ctx = load_schema("./dblp_schema.rdf", base_iri="https://dblp.org/rdf/schema#")

# Mention Extraction

In [ ]:
from pydantic import BaseModel, Field
from typing import Dict, List, Optional

# ----------------------------
# Prompt (focus on predicate CURIE, ignore predicate label column)
# ----------------------------

def build_extractor_prompt(ctx: SchemaIndex) -> str:
    ctx_me = filter_schema(
        ctx,
        keep_only_literal_or_labelable_object=True,
        keep_only_classes_with_predicates=True,
    )

    string_ctx = schema_to_context_string(
        ctx_me,
        include_prefixes=False,
        max_types=100,
        max_preds_per_type=100,
        include_notes=True,
    )

    return f"""You are an entity mention extractor for a knowledge graph.

Goal
Extract entity mentions from the user query and select the correct STRING-LABEL predicate
to later retrieve matching entities from the knowledge graph.

Hard rules
- Only output spans that are contiguous substrings of the user text. NEVER invent text.
- Every mention text and every attr value MUST appear verbatim in the user query.
- Do not infer or complete missing information.
- Do not output duplicate (text, type) pairs.
- Return ONLY valid JSON.

How to choose `type`
- `type` MUST be exactly one of the Types listed in Schema Context.
- Output the TYPE CURIE exactly as shown (the left of the "|"), NOT the label text.

How to choose `label_pred` (CRITICAL)
- `label_pred` MUST be exactly one predicate CURIE listed under the chosen `type`.
- IMPORTANT: In Schema Context predicate rows have the shape:
    pred_curie | pred_label | rng:...
  You MUST output the pred_curie (left side). Ignore pred_label entirely.
- `label_pred` MUST have rng:lit(xsd:string).
- Choose the predicate whose VALUES are most likely to contain the mention text verbatim.
- If multiple predicates are plausible, pick the best one.

Attrs
- attrs are optional metadata, NOT part of the mention text.
- Only use attribute keys that appear in Schema Context for that type.
- Values MUST be verbatim substrings of the user query.
- Do not emit empty strings.
- Year ranges:
  If a range appears (e.g., "2015-2020" or "between 2015 and 2020"),
  store it on ONE mention as "year_start":"2015","year_end":"2020".

Schema Context
{string_ctx}

Output JSON schema:
{{"mentions":[{{"text":str,"type":str,"label_pred":str,"attrs":{{str:str}}}}]}}
Return only JSON, no prose.
"""

class Mention(BaseModel):
    text: str
    type: str
    label_pred: str = Field(description="The predicate label that indicates the type of this mention")
    attrs: Dict[str, str] = Field(default_factory=dict)
    
class MentionList(BaseModel):
    mentions: List[Mention]

def _to_iri(term: str, ns: Dict[str, str]) -> Optional[str]:
    term = (term or "").strip()
    if not term:
        return None
    if term.startswith("<") and term.endswith(">"):
        return term[1:-1]
    if "://" in term:
        return term
    if ":" in term:
        pfx, local = term.split(":", 1)
        if pfx in ns:
            return ns[pfx] + local
    return None

def validate_types(mentions: List[Mention], ctx: "SchemaIndex") -> MentionList:
    """
    Minimal:
      - type must be a known class (CURIE or IRI)
      - label_pred must be a known prop (CURIE or IRI)
      - prop must have type in its domain
      - prop must be literal + include xsd:string in its literal_datatypes
    Invalid mentions are dropped.
    """
    ns = ctx.namespaces
    out: List[Mention] = []

    for m in mentions:
        t = _to_iri(m.type, ns)
        p = _to_iri(m.label_pred, ns)
        if not t or t not in ctx.classes:
            print(f"Skipping mention with unknown type: {m.type}")
            continue
        if not p or p not in ctx.props:
            print(f"Skipping mention with unknown prop: {m.label_pred}")
            continue

        pd = ctx.props[p]
        if t not in pd.domains:
            print(f"Skipping mention with prop {m.label_pred} not valid for type {m.type}")
            continue

        ok_string = any(dt.endswith("#string") for dt in pd.literal_datatypes)
        if not ok_string:
            print(f"Skipping mention with prop {m.label_pred} not literal xsd:string")
            continue

        out.append(Mention(text=m.text, type=t, label_pred=p, attrs=dict(m.attrs)))

    return mentions
    
    
def sanitize_response(content: str) -> str:
    match = re.search(r'\{.*\}', content, re.DOTALL)
    if match:
        return match.group(0)
    print(content)
    raise ValueError("No JSON object found in the response")

async def extract_mentions(q: str, prompt: str, model: str = "llama3.3:70b") -> MentionList:
    resp = await client.chat(
        model=model,
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": q}
        ],
    )
    # debug = resp.model_dump()
    # debug.pop("message", None)
    # print("Debug Response:", debug)
    ml = MentionList.model_validate_json(sanitize_response(resp.message.content))
    return ml.mentions

In [18]:
prompt = build_extractor_prompt(ctx)
print(len(prompt))
print(prompt)

7193
You are an entity mention extractor for a knowledge graph.

Goal
Extract entity mentions from the user query and select the correct STRING-LABEL predicate
to later retrieve matching entities from the knowledge graph.

Hard rules
- Only output spans that are contiguous substrings of the user text. NEVER invent text.
- Every mention text and every attr value MUST appear verbatim in the user query.
- Do not infer or complete missing information.
- Do not output duplicate (text, type) pairs.
- Return ONLY valid JSON.

How to choose `type`
- `type` MUST be exactly one of the Types listed in Schema Context.
- Output the TYPE CURIE exactly as shown (the left of the "|"), NOT the label text.

How to choose `label_pred` (CRITICAL)
- `label_pred` MUST be exactly one predicate CURIE listed under the chosen `type`.
- IMPORTANT: In Schema Context predicate rows have the shape:
    pred_curie | pred_label | rng:...
  You MUST output the pred_curie (left side). Ignore pred_label entirely.
- `lab

In [19]:
for q in queries:
    if "mentions" in data[q]:
        del data[q]["mentions"]

In [20]:
for q in queries[:]:
    if data[q].get('mentions'):
        continue
    print(f"\nQuery: {q}")
    mentions = await extract_mentions(q, prompt, "gpt-oss:120b")
    print("Extracted Mentions:")
    for m in mentions:
        print(f"  {m.type} {m.label_pred}: {m.text}, attrs={m.attrs}")
    data[q]['mentions'] = mentions



Query: show me citations for the paper "Attention Is All You Need" by Vaswani et al 2017
Extracted Mentions:
  dblp:Publication dblp:title: Attention Is All You Need, attrs={'yearOfPublication': '2017'}
  dblp:Creator dblp:creatorName: Vaswani, attrs={}

Query: list all papers by Yoshua Bengio presented at neurips between 2015 and 2020
Extracted Mentions:
  dblp:Creator dblp:creatorName: Yoshua Bengio, attrs={}
  dblp:Stream dblp:streamTitle: neurips, attrs={'year_start': '2015', 'year_end': '2020'}

Query: who are the authors of the iclr 2021 paper "self-supervised learning is all you need"?
Extracted Mentions:
  dblp:Stream dblp:streamTitle: iclr, attrs={}
  dblp:Publication dblp:title: self-supervised learning is all you need, attrs={'yearOfPublication': '2021'}

Query: give me the affiliation of the author named Raquel Urtasun in icml publications
Extracted Mentions:
  dblp:Creator dblp:affiliation: Raquel Urtasun, attrs={}
  dblp:Stream dblp:streamTitle: icml, attrs={}

Query: li

In [23]:
for q in queries[:]:
    mentions = data[q].get('mentions', [])
    print(f"\nQuery: {q}")
    print("Extracted Mentions:")
    for m in mentions:
        print(f"  {m.type} {m.label_pred}: {m.text}, attrs={m.attrs}")

    valid_mentions = validate_types(mentions, ctx)
    if len(valid_mentions) != len(mentions):
        print("Detected invalid mentions.")


Query: show me citations for the paper "Attention Is All You Need" by Vaswani et al 2017
Extracted Mentions:
  dblp:Publication dblp:title: Attention Is All You Need, attrs={'yearOfPublication': '2017'}
  dblp:Creator dblp:creatorName: Vaswani, attrs={}

Query: list all papers by Yoshua Bengio presented at neurips between 2015 and 2020
Extracted Mentions:
  dblp:Creator dblp:creatorName: Yoshua Bengio, attrs={}
  dblp:Stream dblp:streamTitle: neurips, attrs={'year_start': '2015', 'year_end': '2020'}

Query: who are the authors of the iclr 2021 paper "self-supervised learning is all you need"?
Extracted Mentions:
  dblp:Stream dblp:streamTitle: iclr, attrs={}
  dblp:Publication dblp:title: self-supervised learning is all you need, attrs={'yearOfPublication': '2021'}

Query: give me the affiliation of the author named Raquel Urtasun in icml publications
Extracted Mentions:
  dblp:Creator dblp:affiliation: Raquel Urtasun, attrs={}
  dblp:Stream dblp:streamTitle: icml, attrs={}

Query: li

# Get candidate entities for a mention

In [ ]:
from collections import defaultdict
from typing import Dict, List, Optional
import re

# ============================================================
# small helpers
# ============================================================

def _values_block(var: str, terms: list[str]) -> str:
    if not terms:
        return f"VALUES ?{var} {{}}"
    items = []
    for t in terms:
        t = (t or "").strip()
        if not t:
            continue
        if t.startswith("<") and t.endswith(">"):
            items.append(t)
        elif "://" in t:
            items.append(f"<{t}>")
        elif ":" in t:
            items.append(t)  # CURIE
        else:
            items.append(f"<{t}>")
    return f"VALUES ?{var} {{ {' '.join(items)} }}"


def cell(v):
    return v.get("value") if isinstance(v, dict) else v


def normalize_rows(rows, role_name: str, mention_text: str):
    out = []
    for r in rows:
        s = cell(r.get("s"))
        p = cell(r.get("p"))
        lbl = cell(r.get("label"))

        try:
            exact = int(float(cell(r.get("exact")))) if r.get("exact") else 0
        except Exception:
            exact = 0
        if not exact and isinstance(lbl, str):
            exact = int(lbl.lower() == (mention_text or "").lower())

        try:
            label_len = int(float(cell(r.get("labelLen")))) if r.get("labelLen") else (
                len(lbl) if isinstance(lbl, str) else 0
            )
        except Exception:
            label_len = len(lbl) if isinstance(lbl, str) else 0

        out.append(
            {
                "uri": s,
                "pred": p,
                "label": lbl,
                "role": role_name,
                "match_exact": bool(exact),
                "label_len": label_len,
            }
        )
    return out


def merge_by_uri(items):
    grouped = defaultdict(list)
    for it in items:
        grouped[it["uri"]].append(it)
    return [{"uri": uri, "variants": variants} for uri, variants in grouped.items()]


# ============================================================
# SchemaIndex helpers
# ============================================================

def _curie_to_iri(term: str, namespaces: Dict[str, str]) -> str:
    if term.startswith("<") and term.endswith(">"):
        return term[1:-1]
    if "://" in term:
        return term
    if ":" not in term:
        return term
    pfx, suf = term.split(":", 1)
    ns = namespaces.get(pfx)
    return (ns + suf) if ns else term


def _is_xsd(dt_iri: str) -> bool:
    return dt_iri.startswith("http://www.w3.org/2001/XMLSchema#")


def _xsd_local(dt_iri: str) -> str:
    return dt_iri.rsplit("#", 1)[-1] if "#" in dt_iri else dt_iri.rsplit("/", 1)[-1]


def _type_iri_from_curie(idx, type_curie: str) -> str:
    return _curie_to_iri(type_curie, idx.namespaces)


def _is_string_label_pred(idx, type_curie: str, pred_curie: str) -> bool:
    t_iri = _curie_to_iri(type_curie, idx.namespaces)
    p_iri = _curie_to_iri(pred_curie, idx.namespaces)

    if t_iri not in idx.classes:
        return False
    if p_iri not in idx.props:
        return False

    pd = idx.props[p_iri]
    if t_iri not in (pd.domains or set()):
        return False

    if not pd.literal_datatypes:
        return False

    return any(
        _is_xsd(dt) and _xsd_local(dt).lower() == "string"
        for dt in pd.literal_datatypes
    )


# ============================================================
# Candidate query — USE mention.label_pred (single predicate)
# ============================================================

def build_candidate_query_from_mention(
    idx,
    mention,
    limit: int = 50,
) -> str:
    """
    Build candidate query using:
      - mention.type        -> class
      - mention.label_pred  -> EXACT predicate to match labels
    """

    if not _is_string_label_pred(idx, mention.type, mention.label_pred):
        raise ValueError(
            f"Invalid label_pred for type: {mention.type} / {mention.label_pred}"
        )

    q_esc = (mention.text or "").strip("\"").replace('"', '\\"')

    vals_cls = _values_block("cls", [mention.type])
    vals_p = _values_block("p", [mention.label_pred])

    prefixes = "\n".join(
        f"PREFIX {pfx}: <{ns}>"
        for pfx, ns in sorted(idx.namespaces.items())
    )

    return f"""
{prefixes}

SELECT DISTINCT ?s ?p ?label (STRLEN(STR(?label)) AS ?labelLen)
       (IF(LCASE(STR(?label))=LCASE("{q_esc}"), 1, 0) AS ?exact)
WHERE {{
  {vals_cls}
  {vals_p}
  ?s a ?cls ; ?p ?label .
  FILTER(isLiteral(?label))
  FILTER(CONTAINS(LCASE(STR(?label)), LCASE("{q_esc}")))
}}
LIMIT {int(limit)}
""".strip()


def get_candidates_from_mention(
    idx,
    endpoint: str,
    mention,
    limit: int = 25,
    *,
    run_fn,
):
    query = build_candidate_query_from_mention(idx, mention, limit=limit)
    rows = run_fn(endpoint, query)
    return normalize_rows(
        rows,
        role_name=mention.type,
        mention_text=mention.text,
    )


# ============================================================
# refinement — literal attrs only, schema-driven
# ============================================================


def refine_query_with_attrs(idx: SchemaIndex, og_query: str, mention: Mention) -> str:
    if not mention.attrs:
        return og_query

    class_iri = _type_iri_from_curie(idx, mention.type)

    # only literal predicates on this class
    literal_props = []
    for p in idx.props.values():
        if class_iri in (p.domains or set()) and p.literal_datatypes:
            literal_props.append(p)

    def _norm(s: str) -> str:
        return re.sub(r"[^a-z0-9]", "", (s or "").lower())

    # map key -> predicate CURIE
    key_to_pred: Dict[str, str] = {}
    for p in literal_props:
        p_curie = make_curie(p.iri, idx.namespaces)
        if getattr(p, "labels", None):
            for lab in p.labels:
                key_to_pred[_norm(lab)] = p_curie
        key_to_pred[_norm(p_curie.split(":", 1)[-1])] = p_curie

    triples = []
    filters = []

    for k, v in (mention.attrs or {}).items():
        p_curie = key_to_pred.get(_norm(k))
        if not p_curie:
            continue

        var = f"?v_{_norm(k) or 'attr'}"
        triples.append(f"?s {p_curie} {var} .")

        v_esc = str(v).replace('"', '\\"')
        filters.append(f'FILTER(CONTAINS(LCASE(STR({var})), LCASE("{v_esc}")))')

    if not triples:
        return og_query

    insert_block = "\n  " + "\n  ".join(triples + filters) + "\n"

    q = og_query

    # Insert before the closing brace of WHERE that is followed by query modifiers
    m = re.search(r"\}\s*(ORDER\s+BY|GROUP\s+BY|HAVING|LIMIT|OFFSET)\b", q, flags=re.IGNORECASE)
    if m:
        brace_pos = m.start()
        return q[:brace_pos] + insert_block + q[brace_pos:]

    # Fallback: insert before the last '}' in the query
    last = q.rfind("}")
    if last != -1:
        return q[:last] + insert_block + q[last:]

    return og_query

In [42]:
mention = data[queries[0]]['mentions'][0]
mention

Mention(text='Attention Is All You Need', type='dblp:Publication', label_pred='dblp:title', attrs={'yearOfPublication': '2017'})

In [51]:
base_q1 = build_candidate_query_from_mention(ctx, mention, 30)
# print(base_q1)

In [52]:
# res = run(ENDPOINT, base_q1)
# res

In [61]:
q1 = refine_query_with_attrs(ctx, base_q1, mention)

In [57]:
# print(sanitize_query(q1))

In [62]:
res = run(ENDPOINT, q1)
res

[{'s': {'type': 'uri',
   'value': 'https://dblp.org/rec/conf/nips/VaswaniSPUJGKP17'},
  'p': {'type': 'uri', 'value': 'https://dblp.org/rdf/schema#title'},
  'label': {'type': 'literal', 'value': 'Attention is All you Need.'},
  'labelLen': {'datatype': 'http://www.w3.org/2001/XMLSchema#int',
   'type': 'literal',
   'value': '26'},
  'exact': {'datatype': 'http://www.w3.org/2001/XMLSchema#int',
   'type': 'literal',
   'value': '0'}},
 {'s': {'type': 'uri',
   'value': 'https://dblp.org/rec/journals/corr/VaswaniSPUJGKP17'},
  'p': {'type': 'uri', 'value': 'https://dblp.org/rdf/schema#title'},
  'label': {'type': 'literal', 'value': 'Attention Is All You Need.'},
  'labelLen': {'datatype': 'http://www.w3.org/2001/XMLSchema#int',
   'type': 'literal',
   'value': '26'},
  'exact': {'datatype': 'http://www.w3.org/2001/XMLSchema#int',
   'type': 'literal',
   'value': '0'}}]

In [65]:
# norm = normalize_rows(res, role_name=mention.type, mention_text=mention.text)  
# merged = merge_by_uri(norm)   
# merged

In [66]:
async def get_candidates_with_fallback(ctx: SchemaIndex, mention: Mention, limit: int = 30):
    """
    1) Run base candidate query.
    2) If it errors (timeout/400/whatever) OR returns exactly 'limit' items AND we have attrs,
       call LLM to refine the query and retry once.
    3) Return normalized items (not merged).
    """
    role_name = mention.type
    base_q = build_candidate_query_from_mention(ctx, mention, 30)

    # Run base query
    try:
        base_rows = run(ENDPOINT, base_q)
        items = normalize_rows(base_rows, role_name, mention.text)
        saturated = (len(items) >= limit)
        if saturated and mention.attrs:  # force refinement only when we have attrs
            print(f"Refining query for mention {mention.text} with attrs {mention.attrs}")
            refined_q = await refine_query_with_attrs(ctx, base_q, mention)
            print(f"Refined query:\n{sanitize_query(refined_q)}\n")
            refined_rows = run(ENDPOINT, refined_q)
            return normalize_rows(refined_rows, role_name, mention.text)
        return items
    except Exception as e:
        # If base failed and we have attrs, try refined once
        if mention.attrs:
            try:
                refined_q = refine_query_with_attrs(ctx, base_q, mention)
                refined_rows = run(ENDPOINT, refined_q)
                return normalize_rows(refined_rows, role_name, mention.text)
            except Exception:
                # fall through to empty list on double failure
                return []
        # No attrs -> nothing else to try
        return []


In [67]:
# Get candidates per mention with context pack
for query in data:
    mentions = data[query].get("mentions", [])
    if not mentions:
        continue

    print("\n" + "=" * 80)
    print(f"Query: {query}")
    # if len(data[query].get("mentions_with_candidates", [])) == len(mentions):
    #     print(" Skipping, candidates already fetched.")
    #     continue
    data[query]["mentions_with_candidates"] = []

    for m in mentions:
        print(f"\nProcessing mention: {m.text}  (Type: {m.type})")
        entry = {"mention": m, "query": query, "candidates": []}
        try:
            cand_items = await get_candidates_with_fallback(ctx, m, limit=30)
            entry["candidates"] = merge_by_uri(cand_items)
            print(f" → Found {len(cand_items)} candidates, {len(entry['candidates'])} unique URIs")
        except Exception as e:
            print(f" ⚠️  Error fetching candidates: {e}")
            entry["candidates"] = []
        data[query]["mentions_with_candidates"].append(entry)



Query: show me citations for the paper "Attention Is All You Need" by Vaswani et al 2017

Processing mention: Attention Is All You Need  (Type: dblp:Publication)
Refining query for mention Attention Is All You Need with attrs {'yearOfPublication': '2017'}
 → Found 2 candidates, 2 unique URIs

Processing mention: Vaswani  (Type: dblp:Creator)
 → Found 21 candidates, 21 unique URIs

Query: list all papers by Yoshua Bengio presented at neurips between 2015 and 2020

Processing mention: Yoshua Bengio  (Type: dblp:Creator)
 → Found 1 candidates, 1 unique URIs

Processing mention: neurips  (Type: dblp:Stream)
 → Found 3 candidates, 3 unique URIs

Query: who are the authors of the iclr 2021 paper "self-supervised learning is all you need"?

Processing mention: iclr  (Type: dblp:Stream)
 → Found 1 candidates, 1 unique URIs

Processing mention: self-supervised learning is all you need  (Type: dblp:Publication)
 → Found 1 candidates, 1 unique URIs

Query: give me the affiliation of the author n

# Enriching candidates with context

In [77]:
def render_prefixes(ctx) -> str:
    """
    Render PREFIX declarations from ctx.namespaces
    ctx.namespaces: Dict[str, str]  (prefix -> IRI)
    """
    return "\n".join(
        f"PREFIX {p}: <{iri}>"
        for p, iri in ctx.namespaces.items()
    )

# --- 1. Literals only ------------------------------------------------------

def build_onehop_literals(ctx, iri: str, limit: int = 50) -> str:
    return f"""{render_prefixes(ctx)}

SELECT ?p (STR(?o) AS ?value)
WHERE {{
  VALUES ?s {{ <{iri}> }}
  ?s ?p ?o .
  MINUS {{ ?s rdf:type ?o }}
  FILTER(isLiteral(?o))
}}
LIMIT {int(limit)}
""".strip()


# --- 2. IRI neighbors (labels preferred) -----------------------------------

def build_onehop_iris(ctx, iri: str, limit: int = 50) -> str:
    return f"""{render_prefixes(ctx)}

SELECT ?p ?value
WHERE {{
  VALUES ?s {{ <{iri}> }}
  ?s ?p ?o .
  MINUS {{ ?s rdf:type ?o }}
  FILTER(isIRI(?o))

  OPTIONAL {{ ?o rdfs:label  ?lab  . FILTER(isLiteral(?lab)) }}
  OPTIONAL {{ ?o foaf:name   ?nm   . FILTER(isLiteral(?nm))  }}
  OPTIONAL {{ ?o schema:name ?snm  . FILTER(isLiteral(?snm)) }}

  BIND(COALESCE(?lab, ?nm, ?snm, STR(?o)) AS ?value)
}}
LIMIT {int(limit)}
""".strip()


# --- 3. Incoming authoredBy edges (papers for a person) --------------------


def build_onehop_iris_in(ctx, iri: str, limit: int = 50) -> str:
    return f"""{render_prefixes(ctx)}

SELECT ?p ?value
WHERE {{
  VALUES ?center {{ <{iri}> }}
  ?s ?p ?center .
  FILTER(isIRI(?s))

  OPTIONAL {{ ?s rdfs:label  ?lab  . FILTER(isLiteral(?lab)) }}
  OPTIONAL {{ ?s foaf:name   ?nm   . FILTER(isLiteral(?nm))  }}
  OPTIONAL {{ ?s schema:name ?snm  . FILTER(isLiteral(?snm)) }}

  BIND(COALESCE(?lab, ?nm, ?snm, STR(?s)) AS ?value)
}}
LIMIT {int(limit)}
""".strip()


# --- 4. Combined readable one hop -----------------------------------------

def onehop_readable(ctx, iri: str, limit_each: int = 50):
    """Fetch one hop triples around iri and merge into a clean list."""
    rows = []
    rows += run(ENDPOINT, build_onehop_literals(ctx, iri, limit_each))
    rows += run(ENDPOINT, build_onehop_iris(ctx, iri, limit_each))
    rows += run(ENDPOINT, build_onehop_iris_in(ctx, iri, limit_each))

    def cell(b, k):
        v = b.get(k)
        return v.get("value") if isinstance(v, dict) else v

    out = []
    seen = set()

    for r in rows:
        p = cell(r, "p")
        v = cell(r, "value")
        if not p or not v:
            continue
        key = (p, v)
        if key in seen:
            continue
        seen.add(key)
        out.append({"p": p, "value": v})

    return out

In [79]:
# res = onehop_readable(ctx, "https://dblp.org/pid/26/9012", 100)
# res

In [80]:
# Enrich every candidate with 1-hop readable triples
for query, qdata in data.items():
    mention_entries = qdata.get("mentions_with_candidates", [])    
    if not mention_entries:
        continue

    print("\n" + "=" * 80)
    print(f"Enriching candidates for query: {query}")

    for m_entry in mention_entries:
        mention = m_entry["mention"]
        candidates = m_entry.get("candidates", [])
        if not candidates:
            continue

        print(f"\n→ Mention: {mention.text} ({mention.type}) — {len(candidates)} candidates")

        for c_idx, cand_group in enumerate(candidates, start=1):
            uri = cand_group.get("uri")
            if not uri:
                continue
            if "triples" in cand_group:
                print(f"   [{c_idx}/{len(candidates)}] Skipping {uri}, triples already fetched")
                #continue
            print(f"   [{c_idx}/{len(candidates)}] Fetching 1-hop for {uri}")
            try:
                triples = onehop_readable(ctx, uri, limit_each=100)
                print(triples)
                cand_group["triples"] = triples
                print(f"      → Got {len(triples)} triples")
            except Exception as e:
                print(f"      ⚠️  Error fetching triples for {uri}: {e}")
                cand_group["triples"] = []


Enriching candidates for query: show me citations for the paper "Attention Is All You Need" by Vaswani et al 2017

→ Mention: Attention Is All You Need (dblp:Publication) — 2 candidates
   [1/2] Skipping https://dblp.org/rec/conf/nips/VaswaniSPUJGKP17, triples already fetched
   [1/2] Fetching 1-hop for https://dblp.org/rec/conf/nips/VaswaniSPUJGKP17
[{'p': 'https://dblp.org/rdf/schema#numberOfCreators', 'value': '8'}, {'p': 'https://dblp.org/rdf/schema#pagination', 'value': '5998-6008'}, {'p': 'http://www.w3.org/2000/01/rdf-schema#label', 'value': 'Ashish Vaswani et al.: Attention is All you Need. (2017)'}, {'p': 'https://dblp.org/rdf/schema#title', 'value': 'Attention is All you Need.'}, {'p': 'https://dblp.org/rdf/schema#publishedIn', 'value': 'NIPS'}, {'p': 'https://dblp.org/rdf/schema#publishedInBook', 'value': 'NIPS'}, {'p': 'https://dblp.org/rdf/schema#yearOfEvent', 'value': '2017'}, {'p': 'https://dblp.org/rdf/schema#yearOfPublication', 'value': '2017'}, {'p': 'https://dblp.or

# Ranking Candidates

In [81]:
def simplify_predicate(predicate: str, ctx: SchemaIndex) -> str:
    """
    If predicate is an IRI, try to shorten to CURIE using ctx.namespaces.
    Otherwise return as-is (for CURIEs) or fallback to localname.
    """
    predicate = (predicate or "").strip()
    if not predicate:
        return predicate

    # Already a CURIE
    if ":" in predicate and not predicate.startswith("http"):
        return predicate

    # Try CURIE shortening for full IRIs using namespaces
    if predicate.startswith("http"):
        # prefer longest matching namespace IRI (avoid collisions)
        best_pfx = None
        best_base = None
        for pfx, base in (ctx.namespaces or {}).items():
            if predicate.startswith(base) and (best_base is None or len(base) > len(best_base)):
                best_pfx = pfx
                best_base = base
        if best_pfx and best_base:
            return f"{best_pfx}:{predicate[len(best_base):]}"

        # fallback: localname
        return predicate.rsplit("#", 1)[-1].rsplit("/", 1)[-1]

    return predicate


def candidate_to_sentences(candidate: dict, ctx: SchemaIndex) -> list[str]:
    sentences: list[str] = []

    if "variants" in candidate:
        for variant in candidate["variants"]:
            label = variant.get("label")
            if label:
                sentences.append(f"label {label}")

    if "triples" in candidate:
        for triple in candidate["triples"]:
            pred = triple.get("p", "")
            value = triple.get("value", "")
            if pred and value:
                simple_pred = simplify_predicate(pred, ctx)
                sentences.append(f"{simple_pred} {value}")

    return sentences


def candidate_to_text(candidate: dict, ctx: SchemaIndex) -> str:
    return " . ".join(candidate_to_sentences(candidate, ctx))

In [89]:
query = queries[0]
print(query)
entitiy_idx = 1


docs = [
    candidate_to_text(c, ctx)
    for c in data[query]["mentions_with_candidates"][entitiy_idx]["candidates"]
]
for doc in docs:
    print(len(doc), "\tchars:\t", doc)

show me citations for the paper "Attention Is All You Need" by Vaswani et al 2017
10546 	chars:	 label Sharan Vaswani . rdfs:label Sharan Vaswani . dblp:creatorName Sharan Vaswani . dblp:primaryCreatorName Sharan Vaswani . dblp:authoredBy Reza Asad et al.: Fast Convergence of Softmax Policy Mirror Ascent. (2025) . dblp:createdBy Reza Asad et al.: Fast Convergence of Softmax Policy Mirror Ascent. (2025) . dblp:authoredBy Nicolas Loizou et al.: Stochastic Polyak Step-size for SGD: An Adaptive Learning Rate for Fast Convergence. (2021) . dblp:createdBy Nicolas Loizou et al.: Stochastic Polyak Step-size for SGD: An Adaptive Learning Rate for Fast Convergence. (2021) . dblp:createdBy Si Yi Meng et al.: Fast and Furious Convergence: Stochastic Second Order Methods under Interpolation. (2020) . dblp:authoredBy Si Yi Meng et al.: Fast and Furious Convergence: Stochastic Second Order Methods under Interpolation. (2020) . dblp:createdBy Sharan Vaswani et al.: Fast and Faster Convergence of SGD f

## Reranking with CrossEncoders

In [90]:
from zeroentropy import ZeroEntropy

zclient = ZeroEntropy()

In [91]:
# from sentence_transformers import CrossEncoder

# model = CrossEncoder("zeroentropy/zerank-1-small", trust_remote_code=True)

# print(len(query), "chars in query")
# print( [len(d) for d in docs], "chars in docs")

In [92]:
# query_documents = [
#    (query, doc) for doc in docs[1:3]
# ]
# query_documents

In [93]:
# scores = model.predict(query_documents)
# print(scores)

In [94]:
# response = zclient.models.rerank(
#     model="zerank-1-small",  # Can also use "zerank-1-small"
#     query=query,
#     documents=docs,
# )
# for res in response.results:
#     print(f"Score: {res.relevance_score}\nCandidate: {data[query]['mentions_with_candidates'][entitiy_idx]['candidates'][res.index]}\n")


## Compute scores using bm25

In [95]:
import math
import re
from collections import Counter

def bm25_score(query, docs, k1=1.5, b=0.75):
    query_terms = re.findall(r"\w+", query.lower())
    tokenized = [re.findall(r"\w+", d.lower()) for d in docs]
    N = len(docs)
    avgdl = sum(len(d) for d in tokenized) / N

    scores = []
    for i, d in enumerate(tokenized):
        dl = len(d)
        tf = Counter(d)
        score = 0.0
        for t in query_terms:
            n_t = sum(t in doc for doc in tokenized)
            if n_t == 0:
                continue
            idf = math.log((N - n_t + 0.5) / (n_t + 0.5))
            tf_td = tf[t]
            norm = tf_td * (k1 + 1) / (tf_td + k1 * (1 - b + b * dl / avgdl))
            score += idf * norm
        scores.append((score, i))
    return sorted(scores, key=lambda x: x[0], reverse=True)

res = bm25_score(query, docs)

i = 0
for score, idx in res[:5]:
    c = data[query]['mentions_with_candidates'][entitiy_idx]['candidates'][idx]
    print(f"{idx}, Score: {score}\nCandidate:{c['uri']=}, {c['variants'][0]['label']=}\n")
    # print(f"{i+1}. Score: {score:.4f}\n\turi={c.get('uri')}, label={c.get('variants',[{}])[0].get('label')}")
    # i += 1

5, Score: -2.2451215908711264
Candidate:c['uri']='https://dblp.org/pid/26/9012', c['variants'][0]['label']='Ashish Vaswani'

6, Score: -8.192321614230464
Candidate:c['uri']='https://dblp.org/pid/270/9442', c['variants'][0]['label']='Ashwin Vaswani'

8, Score: -8.623946315366743
Candidate:c['uri']='https://dblp.org/pid/284/7374', c['variants'][0]['label']='Chirag Vaswani Bhavnani'

9, Score: -8.699374417250416
Candidate:c['uri']='https://dblp.org/pid/294/6108', c['variants'][0]['label']='Ram Vaswani'

17, Score: -8.820901037928357
Candidate:c['uri']='https://dblp.org/pid/44/5407', c['variants'][0]['label']='Peter K. T. Vaswani'



## Compute scores using embeddings and cosine similarity

In [38]:
import numpy as np

async def embed(text: str):
    res = await client.embed(model="nomic-embed-text:latest", input=text)
    return res.embeddings[0]

def cosine_similarity(a, b):
    """Compute cosine similarity (simplified for normalized vectors)"""
    a = np.array(a)
    b = np.array(b)
    return float(a @ b)  # Just dot product for normalized vectors

async def score_embeddings(query: str, evidence: list[str]):
    if not evidence:
        return 0.0
    
    q_emb = await embed(query)
    scores = []
    for i, s in enumerate(evidence):
        e_emb = await embed(s)
        scores.append((cosine_similarity(q_emb, e_emb), i))
    
    scores.sort(reverse=True, key=lambda x: x[0])
    return scores

In [39]:
scores = await score_embeddings(query, docs)
for score, idx in scores[:]:
    c = data[query]['mentions_with_candidates'][entitiy_idx]['candidates'][idx]
    print(f"{idx}, Score: {score}\nCandidate:{c['uri']=}, {c['variants'][0]['label']=}\n")

1, Score: 0.7664981657141212
Candidate:c['uri']='https://dblp.org/rec/journals/corr/VaswaniSPUJGKP17', c['variants'][0]['label']='Ashish Vaswani et al.: Attention Is All You Need. (2017)'

0, Score: 0.46406496048508605
Candidate:c['uri']='https://dblp.org/rec/conf/nips/VaswaniSPUJGKP17', c['variants'][0]['label']='Ashish Vaswani et al.: Attention is All you Need. (2017)'



## jina-reranker-v2


In [ ]:
# !pip install transformers einops

In [ ]:
# from transformers import AutoModelForSequenceClassification

# model = AutoModelForSequenceClassification.from_pretrained(
#     'jinaai/jina-reranker-v2-base-multilingual',
#     dtype="auto",
#     trust_remote_code=True,
#     use_flash_attn=False
# )

# model.to('cpu') # or 'cpu' if no GPU is available
# model.eval()


In [ ]:
# construct sentence pairs
# sentence_pairs = [[query, doc] for doc in docs]
# scores = model.compute_score(sentence_pairs, max_length=1024)


In [ ]:
# score_list = scores.tolist()

